# Detecção de Anomalias em Transações em Python

##OBJETIVO:

- Identificar as possíveis transações fraudulentas com o uso do cartão de crédito

## IMPORTAÇÃO DO DATASET

In [ ]:
import pandas as pd

link='https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'

df=pd.read_csv(link)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [ ]:
df.shape

(284807, 31)

In [ ]:
df.tail()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0
284806,172792.0,-0.533413,-0.189733,0.703337,-0.506271,-0.012546,-0.649617,1.577006,-0.414650,0.486180,...,0.261057,0.643078,0.376777,0.008797,-0.473649,-0.818267,-0.002415,0.013649,217.00,0


In [ ]:
df['Class'].value_counts(normalize=True)

,proportion
Class,
0,0.998273
1,0.001727


## Observações:

- O dataset apresenta-se limpo de dados nulos;
- V1, V2, ..., V28 são os componentes principais obtidos com a PCA;
- A característica 'Class' é a variável resposta e assume o valor 1 em caso de fraude e 0 caso contrário;
- A proporcionalidade da variável 'Class' apresenta-se mais concentrado em transações não fraudulentas 99.83%, enquanto, as fraudulentas 0.17%, ou seja, o conjunto de dados do alvo 'Class' encontram-se desbalanceados para treinar o modelo de Machine Learning.

## Feature Enginnering

In [ ]:
import numpy as np

df['Amount_log']=np.log1p(df['Amount'])

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df["Amount_scaled"] = scaler.fit_transform(df[["Amount"]])

## Treinamento dos Modelos

In [ ]:
#Separação dos dados para os modelos
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE


X=df.drop(['Class'],axis=1)
y=df[['Class']]

X_train,X_test,y_train,y_test=train_test_split(X,
                                               y,
                                               stratify=y,
                                               test_size=0.3,
                                               random_state=43)

#Equlibribrando a variável 'Class'
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X, y)

# Modelos

## Modelo LogisticRegression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


modelo_lor=LogisticRegression(max_iter=1000)
modelo_lor.fit(X_train_res,y_train_res)
y_pred_lor=modelo_lor.predict(X_test)

print(classification_report(y_test, y_pred_lor))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


              precision    recall  f1-score   support

           0       1.00      0.99      0.99     85295
           1       0.12      0.92      0.22       148

    accuracy                           0.99     85443
   macro avg       0.56      0.95      0.61     85443
weighted avg       1.00      0.99      0.99     85443



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
modelo_lor.fit(X_train,y_train)
y_pred_lor=modelo_lor.predict(X_test)
print(classification_report(y_test, y_pred_lor))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.85      0.66      0.74       148

    accuracy                           1.00     85443
   macro avg       0.93      0.83      0.87     85443
weighted avg       1.00      1.00      1.00     85443



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Modelo RandomForestClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelo_rfc = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

modelo_rfc.fit(X_train_res,y_train_res)
y_pred_rf = modelo_rfc.predict(X_test)
print(classification_report(y_test, y_pred_rf))

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.61      0.96      0.75       148

    accuracy                           1.00     85443
   macro avg       0.80      0.98      0.87     85443
weighted avg       1.00      1.00      1.00     85443



In [ ]:
modelo_rfc.fit(X_train,y_train)
y_pred_rf = modelo_rfc.predict(X_test)
print(classification_report(y_test, y_pred_rf))

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.83      0.81      0.82       148

    accuracy                           1.00     85443
   macro avg       0.92      0.91      0.91     85443
weighted avg       1.00      1.00      1.00     85443



## Modelo XGBClassifier

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    scale_pos_weight=10,
    use_label_encoder=False,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print(classification_report(y_test, y_pred_xgb))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:03:36] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.90      0.82      0.86       148

    accuracy                           1.00     85443
   macro avg       0.95      0.91      0.93     85443
weighted avg       1.00      1.00      1.00     85443



## Modelo KNeighborsClassifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

modelo_kng=KNeighborsClassifier()
modelo_kng.fit(X_train_res,y_train_res)
y_pred_kng = modelo_kng.predict(X_test)
print(classification_report(y_test, y_pred_kng))

/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


              precision    recall  f1-score   support

           0       1.00      0.97      0.98     85295
           1       0.05      0.99      0.10       148

    accuracy                           0.97     85443
   macro avg       0.53      0.98      0.54     85443
weighted avg       1.00      0.97      0.98     85443



In [ ]:
modelo_kng.fit(X_train,y_train)
y_pred_kng = modelo_kng.predict(X_test)
print(classification_report(y_test, y_pred_kng))

/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.92      0.07      0.14       148

    accuracy                           1.00     85443
   macro avg       0.96      0.54      0.57     85443
weighted avg       1.00      1.00      1.00     85443



# Conclusão:

Com base nos experimentos realizados para a detecção de fraudes em cartões de crédito, conclui-se que o XGBClassifier apresentou o melhor desempenho geral entre todos os algoritmos testados.

O principal desafio do conjunto de dados foi o desbalanceamento extremo da variável resposta (Class), que conta com apenas 0,17% de transações fraudulentas. Estratégias como o uso do SMOTE ajudaram a elevar o Recall (sensibilidade) em modelos como Logistic Regression e KNN, mas geraram um volume altíssimo de falsos positivos, derrubando drasticamente a Precisão.

O XGBoost destacou-se justamente por contornar esse problema de forma eficiente: ao utilizar o parâmetro scale_pos_weight=10, o modelo conseguiu equilibrar o aprendizado sem a necessidade de gerar dados sintéticos. Ele alcançou o maior F1-Score (0.86), combinando uma Precisão de 0.90 (evitando alarmes falsos para clientes legítimos) com um Recall de 0.82 (capturando a grande maioria das fraudes reais).